In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install \
    datasets==2.20.0 \
    pyarrow==16.1.0 \
    fsspec==2024.2.0 \
    huggingface-hub==0.23.4 \
    -q

In [ ]:
import pyarrow
import datasets
import sys

print("Python  :", sys.version)
print("PyArrow :", pyarrow.__version__)
print("Datasets:", datasets.__version__)

# Quick sanity test
try:
    import pyarrow.parquet
    print("PyArrow parquet : OK")
except Exception as e:
    print("PyArrow parquet : FAILED —", e)

In [ ]:
from datasets import load_dataset

massive_val = load_dataset(
    "AmazonScience/massive",
    "hi-IN",
    split='validation',
    trust_remote_code=True
)
massive_test = load_dataset(
    "AmazonScience/massive",
    "hi-IN",
    split='test',
    trust_remote_code=True
)

print(f"Validation: {len(massive_val)}")
print(f"Test: {len(massive_test)}")

In [ ]:
df_massive_val = pd.DataFrame({
    'text'  : massive_val['utt'],
    'intent_num': massive_val['intent']
})
print("Available intents in MASSIVE:")
print(df_massive_val['intent_num'].dtype)
print(df_massive_val['intent_num'].unique())

In [ ]:
df_massive_test = pd.DataFrame({
    'text'  : massive_test['utt'],
    'intent_num': massive_test['intent']
})
print("Available intents in MASSIVE:")
print(df_massive_test['intent_num'].dtype)
print(df_massive_test['intent_num'].unique())

In [ ]:
intent_labels_val = massive_val.features['intent'].names
print("All unique intent labels:")
for i, label in enumerate(intent_labels_val):
    print(f"  {i} : {label}")

In [ ]:
intent_labels_test = massive_test.features['intent'].names
print("All unique intent labels:")
for i, label in enumerate(intent_labels_test):
    print(f"  {i} : {label}")

In [ ]:
df_massive_val['intent'] = df_massive_val['intent_num'].apply(
    lambda x: intent_labels_val[x]
)

df_massive_val[['text', 'intent_num', 'intent']].head()

In [ ]:
df_massive_test['intent'] = df_massive_test['intent_num'].apply(
    lambda x: intent_labels_test[x]
)

df_massive_test[['text', 'intent_num', 'intent']].head()

In [ ]:
intent_mapping = {
    # Reminders
    'alarm_set'          : 'set_reminder',

    # Cancel / remove
    'alarm_remove'       : 'cancel',
    'calendar_remove'    : 'cancel',
    'lists_remove'       : 'cancel',

    # Meetings / calendar
    'calendar_set'       : 'schedule_meeting',

    # Retrieve info
    'alarm_query'        : 'retrieve_info',
    'calendar_query'     : 'retrieve_info',
    'datetime_query'     : 'retrieve_info',
    'datetime_convert'   : 'retrieve_info',
    'lists_query'        : 'retrieve_info',

    # Answer query
    'qa_factoid'         : 'answer_query',
    'qa_definition'      : 'answer_query',
    'qa_currency'        : 'answer_query',
    'qa_stock'           : 'answer_query',
    'qa_maths'           : 'answer_query',
    'general_quirky'     : 'answer_query',
    'general_joke'       : 'answer_query',
    'weather_query'      : 'answer_query',

    # Notes
    'lists_createoradd'  : 'save_note',

    # Greet
    'general_greet'      : 'greet',
}

df_filtered_val = df_massive_val[
    df_massive_val['intent'].isin(intent_mapping.keys())
].copy()

df_filtered_val['intent'] = df_filtered_val['intent'].map(intent_mapping)
df_filtered_val = df_filtered_val[['text', 'intent']]
df_filtered_val['source'] = 'massive'

print(f"Total rows after filtering in val: {len(df_filtered_val)}")
print("\nBasanti intent distribution:")
print(df_filtered_val['intent'].value_counts())


df_filtered_test = df_massive_test[
    df_massive_test['intent'].isin(intent_mapping.keys())
].copy()

df_filtered_test['intent'] = df_filtered_test['intent'].map(intent_mapping)
df_filtered_test = df_filtered_test[['text', 'intent']]
df_filtered_test['source'] = 'massive'

print(f"Total rows after filtering in test: {len(df_filtered_test)}")
print("\nBasanti intent distribution:")
print(df_filtered_test['intent'].value_counts())

In [ ]:
clinic = load_dataset("clinc_oos", "plus", trust_remote_code=True)

df_clinic_val = pd.DataFrame(clinic['validation'])
df_clinic_test = pd.DataFrame(clinic['test'])

print(f"Validation: {len(df_clinic_val)}")
print(f"Test: {len(df_clinic_test)}")

In [ ]:
print("Available intents in CLINIC150:")
print(df_clinic_val['intent'].dtype)
print(df_clinic_val['intent'].unique())

In [ ]:
print("Available intents in CLINIC150:")
print(df_clinic_test['intent'].dtype)
print(df_clinic_test['intent'].unique())

In [ ]:
intent_labels_val = clinic['validation'].features['intent'].names
df_clinic_val['intent_str'] = df_clinic_val['intent'].apply(
    lambda x: intent_labels_val[x]
)

intent_labels_test = clinic['test'].features['intent'].names
df_clinic_test['intent_str'] = df_clinic_test['intent'].apply(
    lambda x: intent_labels_test[x]
)

In [ ]:
df_clinic_val.columns = ['text', 'intent_num', 'intent']
df_clinic_test.columns = ['text', 'intent_num', 'intent']

In [ ]:
clinic_mapping = {
    # Reminders
    'reminder'              : 'set_reminder',
    'reminder_update'       : 'set_reminder',
    'alarm'                 : 'set_reminder',
    'timer'                 : 'set_reminder',

    # Meetings
    'calendar'              : 'schedule_meeting',
    'calendar_update'       : 'schedule_meeting',
    'meeting_schedule'      : 'schedule_meeting',
    'schedule_meeting'      : 'schedule_meeting',

    # Notes
    'todo_list'             : 'save_note',
    'todo_list_update'      : 'save_note',
    'shopping_list'         : 'save_note',
    'shopping_list_update'  : 'save_note',

    # Retrieve info
    'date'                  : 'retrieve_info',
    'time'                  : 'retrieve_info',
    'timezone'              : 'retrieve_info',
    'repeat'                : 'retrieve_info',

    # Answer query
    'meaning_of_life'       : 'answer_query',
    'definition'            : 'answer_query',
    'fun_fact'              : 'answer_query',
    'weather'               : 'answer_query',
    'what_song'             : 'answer_query',
    'measurement_conversion': 'answer_query',
    'calculator'            : 'answer_query',

    # Greet
    'greeting'              : 'greet',
    'goodbye'               : 'greet',
    'thank_you'             : 'greet',

    # Cancel
    'cancel'                : 'cancel',
    'no'                    : 'cancel',

    # Out of scope
    'oos'                   : 'out_of_scope',
}

df_clinic_filtered_val = df_clinic_val[
    df_clinic_val['intent'].isin(clinic_mapping.keys())
].copy()

df_clinic_filtered_val['intent'] = df_clinic_filtered_val['intent'].map(clinic_mapping)
df_clinic_filtered_val = df_clinic_filtered_val[['text', 'intent']]
df_clinic_filtered_val['source'] = 'clinc150'

print(f"Total rows after filtering in val: {len(df_clinic_filtered_val)}")
print(df_clinic_filtered_val['intent'].value_counts())


df_clinic_filtered_test = df_clinic_test[
    df_clinic_test['intent'].isin(clinic_mapping.keys())
].copy()

df_clinic_filtered_test['intent'] = df_clinic_filtered_test['intent'].map(clinic_mapping)
df_clinic_filtered_test = df_clinic_filtered_test[['text', 'intent']]
df_clinic_filtered_test['source'] = 'clinc150'

print(f"Total rows after filtering in test: {len(df_clinic_filtered_test)}")
print(df_clinic_filtered_test['intent'].value_counts())

In [ ]:
import requests

def load_text_from_url(url):
    return requests.get(url).text.strip().split('\n')

# HWU64 test split (from the same GitHub source as your train data)
hwu_test_labels_url = "https://raw.githubusercontent.com/jianguoz/Few-Shot-Intent-Detection/main/Datasets/HWU64/test/label"
hwu_test_seq_url    = "https://raw.githubusercontent.com/jianguoz/Few-Shot-Intent-Detection/main/Datasets/HWU64/test/seq.in"

hwu_test_labels = load_text_from_url(hwu_test_labels_url)
hwu_test_utterances = load_text_from_url(hwu_test_seq_url)

df_hwu_test = pd.DataFrame({'text': hwu_test_utterances, 'intent_str': hwu_test_labels})
print(f"HWU64 test: {len(df_hwu_test)}")

In [ ]:
hwu_mapping = {
    # Reminders
    'alarm_set'            : 'set_reminder',

    # Cancel
    'alarm_remove'         : 'cancel',
    'calendar_remove'      : 'cancel',
    'lists_remove'         : 'cancel',
    'general_negate'       : 'cancel',
    'general_dontcare'     : 'cancel',
    'general_commandstop'  : 'cancel',

    # Meetings
    'calendar_set'         : 'schedule_meeting',

    # Retrieve info
    'alarm_query'           : 'retrieve_info',
    'calendar_query'        : 'retrieve_info',
    'datetime_query'        : 'retrieve_info',
    'datetime_convert'      : 'retrieve_info',
    'lists_query'           : 'retrieve_info',
    'general_repeat'        : 'retrieve_info',

    # Answer query
    'qa_factoid'             : 'answer_query',
    'qa_definition'          : 'answer_query',
    'qa_currency'            : 'answer_query',
    'qa_stock'               : 'answer_query',
    'qa_maths'               : 'answer_query',
    'general_quirky'         : 'answer_query',
    'general_joke'           : 'answer_query',
    'general_explain'        : 'answer_query',
    'weather_query'          : 'answer_query',

    # Notes
    'lists_createoradd'      : 'save_note',
}

In [ ]:
df_hwu_filtered_test = df_hwu_test[
    df_hwu_test['intent_str'].isin(hwu_mapping.keys())
].copy()

df_hwu_filtered_test['intent'] = df_hwu_filtered_test['intent_str'].map(hwu_mapping)
df_hwu_filtered_test = df_hwu_filtered_test[['text', 'intent']]
df_hwu_filtered_test['source'] = 'hwu64'

print(f"Total rows after filtering in test: {len(df_hwu_filtered_test)}")
print(df_hwu_filtered_test['intent'].value_counts())

In [ ]:
df_hwu_filtered_test.head()

In [ ]:
official_val_raw = pd.concat([
    df_filtered_val,
    df_clinic_filtered_val
], ignore_index=True)

official_test_raw = pd.concat([
    df_filtered_test,
    df_clinic_filtered_test,
    df_hwu_filtered_test
], ignore_index=True)


print(f"Total official validation rows (English/Hindi mixed, pre-translation): {len(official_val_raw)}")
print(official_val_raw['intent'].value_counts())
print(official_val_raw.groupby(['intent'])['source'].value_counts())

print(f"Total official test rows (English/Hindi mixed, pre-translation): {len(official_test_raw)}")
print(official_test_raw['intent'].value_counts())
print(official_test_raw.groupby(['intent'])['source'].value_counts())

In [ ]:
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

In [ ]:
!pip install sentencepiece sacremoses -q
!pip install transformers==4.41.2 -q
!pip install IndicTransToolkit -q

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "ai4bharat/indictrans2-en-indic-1B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")

In [ ]:
from IndicTransToolkit.processor import IndicProcessor

ip = IndicProcessor(inference=True)

In [ ]:
def translate_indictrans(texts, batch_size=16):
    translated = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Preprocess
        batch_preprocessed = ip.preprocess_batch(
            batch, src_lang="eng_Latn", tgt_lang="hin_Deva"
        )

        inputs = tokenizer(
            batch_preprocessed,
            truncation=True,
            padding="longest",
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=5,
                max_length=128,
            )

        decoded = tokenizer.batch_decode(
            outputs, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )

        # Postprocess
        postprocessed = ip.postprocess_batch(decoded, lang="hin_Deva")
        translated.extend(postprocessed)

        print(f"Translated {min(i+batch_size, len(texts))}/{len(texts)}")

    return translated

In [ ]:
df_clinic_filtered_val['text_hindi'] = translate_indictrans(df_clinic_filtered_val['text'].tolist())
df_clinic_val_hindi = df_clinic_filtered_val[['text_hindi', 'intent']].copy()
df_clinic_val_hindi.columns = ['text', 'intent']
df_clinic_val_hindi['source'] = 'clinc150'
print(f"Translated {len(df_clinic_filtered_val)} rows")

df_clinic_filtered_test['text_hindi'] = translate_indictrans(df_clinic_filtered_test['text'].tolist())
df_clinic_test_hindi = df_clinic_filtered_test[['text_hindi', 'intent']].copy()
df_clinic_test_hindi.columns = ['text', 'intent']
df_clinic_test_hindi['source'] = 'clinc150'
print(f"Translated {len(df_clinic_filtered_test)} rows")

df_hwu_filtered_test['text_hindi'] = translate_indictrans(df_hwu_filtered_test['text'].tolist())
df_hwu_test_hindi = df_hwu_filtered_test[['text_hindi', 'intent']].copy()
df_hwu_test_hindi.columns = ['text', 'intent']
df_hwu_test_hindi['source'] = 'hwu64'
print(f"Translated {len(df_hwu_filtered_test)} rows")

In [ ]:
df_val = pd.concat([
    df_filtered_val,
    df_clinic_val_hindi
], ignore_index=True)

df_val = df_val.drop_duplicates(subset=['text'])
df_val = df_val.dropna(subset=['text'])
print(f"Total combined rows: {len(df_val)}")
print(df_val['intent'].value_counts())


df_test = pd.concat([
    df_filtered_test,
    df_clinic_test_hindi,
    df_hwu_test_hindi
], ignore_index=True)

df_test = df_test.drop_duplicates(subset=['text'])
df_test = df_test.dropna(subset=['text'])
print(f"Total combined rows: {len(df_test)}")
print(df_test['intent'].value_counts())

In [ ]:
print(df_val.groupby(['intent'])['source'].value_counts())

In [ ]:
print(df_test.groupby(['intent'])['source'].value_counts())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/basanti/datasets', exist_ok=True)

In [ ]:
df_val.to_csv('/content/drive/MyDrive/basanti/datasets/all_sources_combined_val.csv', index=False)
print(f"Saved {len(df_val)} rows")

In [ ]:
df_test.to_csv('/content/drive/MyDrive/basanti/datasets/all_sources_combined_test.csv', index=False)
print(f"Saved {len(df_test)} rows")

In [ ]:
df_val['intent'].value_counts()

In [ ]:
df_test['intent'].value_counts()